# DFT starter — H₂O at STO-3G

End-to-job workflow:

1. Extract basis info for a small molecule.
2. Trace an RHF SCF module with `uniqx`.
3. Call `preflight()` and read the Pareto table.
4. Submit the recommended option and fetch the result.
5. Compare to a PySCF baseline.

Required env vars: `UNIQX_API_KEY`, `UNIQX_GATEWAY`.

## 1 · Setup

In [ ]:
import os

import uniqx
from uniqx.domains.chemistry.basis import extract_basis
from uniqx.domains.chemistry.nmr_full import scf_module

GATEWAY = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
uniqx.login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY)
client = uniqx.connect(GATEWAY)
print("uniqx", uniqx.__version__, "— gateway:", GATEWAY)

## 2 · Workload — H₂O at STO-3G

In [ ]:
H2O = [
    ("O", [0.0, 0.0, 0.1173]),
    ("H", [0.0, 0.7572, -0.4692]),
    ("H", [0.0, -0.7572, -0.4692]),
]

info = extract_basis(H2O, "sto-3g")
print(f"basis functions: {info.n_basis} | electrons: {info.n_electrons} | atoms: {info.n_atoms}")
print(f"nuclear repulsion: {info.nuclear_repulsion:.6f} Ha")

module = scf_module(H2O, info, max_iter=50)
print("traced module:", len(module.functions), "function(s),",
      len(module.functions[0].ops), "ops")

## 3 · Preflight — the Pareto table

In [ ]:
options = uniqx.preflight(module, client=client)
print(options.summary())

Read the table. Pick the option you want to run — the engine's `recommended`,
or override based on what *you* think matters most for this workload.

In [ ]:
choice = options.recommended
print(f"Selected: {choice['label']}")
print(f"  expected time   : {choice.get('total_time', 0)}")
print(f"  expected cost   : ${choice.get('total_cost_usd', 0):.4f}")
print(f"  expected error  : {choice.get('max_error_rate', 0) * 100:.3f}%")
print(f"  expected carbon : {choice.get('total_carbon_g', 0):.3f} g")
print(f"  node assignment : {choice.get('node_assignments', {})}")

## 4 · Submit & retrieve

In [ ]:
runtime_inputs = [
    list(info.exps_flat),
    list(info.coeffs_flat),
    list(info.centers_flat),
    list(info.ang_flat),
    list(info.atom_coords_flat),
    list(info.charges_flat),
]

job_id = uniqx.submit(
    module,
    client=client,
    preflight_job_id=options.job_id,
    option_idx=choice["_idx"],
    runtime_inputs=runtime_inputs,
)
result = uniqx.get(job_id, client=client, timeout=300)

payload = result.get("payload") or result.get("result_payload") or b""
if isinstance(payload, str):
    payload = payload.encode("utf-8", errors="replace")

scf_energy = None
for line in payload.decode("utf-8", errors="replace").splitlines():
    parsed = uniqx.parse_buffer_view(line.strip())
    if parsed is None:
        continue
    _dims, _dtype, vals = parsed
    if vals:
        scf_energy = vals[-1]
        break

print(f"uniqx SCF energy: {scf_energy:.6f} Ha")

## 5 · Baseline — PySCF reference

In [ ]:
from baseline import rhf_reference

ref_energy = rhf_reference(H2O, "sto-3g")
rel_error = abs(scf_energy - ref_energy) / abs(ref_energy)
print(f"PySCF reference : {ref_energy:.6f} Ha")
print(f"uniqx result    : {scf_energy:.6f} Ha")
print(f"relative error  : {rel_error:.2e}")

## 6 · Discussion (your job)

Three prompts for your submission write-up:

1. **Which option would you pick if accuracy mattered 10× more than cost?** Re-read
   the table from cell 3 and justify.
2. **What happens to the frontier if you switch basis to 6-31G?** Re-run cells 2–3
   and paste the new `summary()` next to the original.
3. **What if the molecule grows to methane (CH₄) or methanol (CH₃OH)?** The
   gateway splits the graph differently — watch the `node_assignments` field.